# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Họ và tên:** Nguyễn Quang Đạo  
**MSSV:** 2A202602394  
**Môi trường:** Kaggle Notebooks (GPU Tesla T4 x2 / P100)  

Quy trình thực nghiệm: **Bước 0** (EDA & Sanity check) → **Bước 1** (So sánh 5 backbone) → **Bước 2** (Công thức huấn luyện) → **Bước 3** (Phương pháp suy luận & benchmark độ trễ) → **Bước 4** (Chung kết 3 seed & eval.py) → **Bước 5** (Xuất results.xlsx & báo cáo).

## 0. Cài đặt môi trường & Tải dữ liệu

In [ ]:
# 1. Cài đặt thư viện phụ trợ
!pip install -q timm openpyxl thop

import os, sys, platform, hashlib, time
import numpy as np, pandas as pd, torch, timm

# 2. Tự động clone hoặc cập nhật repo chứa code
REPO_NAME = "K4-Track4-Day2-NguyenQuangDao-2A202602394-Deeplearning-Advance"
REPO_URL = f"https://github.com/nguyenquangdao2004-glitch/{REPO_NAME}.git"

if not os.path.exists(REPO_NAME) and not os.path.exists("dataset.py"):
    print(f"Đang clone repository {REPO_NAME} từ GitHub...")
    !git clone {REPO_URL}
elif os.path.exists(REPO_NAME):
    print("Repo đã tồn tại, cập nhật code mới nhất...")
    !cd {REPO_NAME} && git pull

# 3. Tự động tìm và thêm thư mục chứa code và eval.py vào sys.path
candidate_code_dirs = [
    ".",
    "code",
    f"{REPO_NAME}/submissions/2A202602394_NguyenQuangDao/code",
    f"{REPO_NAME}/code",
    f"{REPO_NAME}",
]

for d in candidate_code_dirs:
    if os.path.exists(os.path.join(d, "dataset.py")):
        abs_d = os.path.abspath(d)
        if abs_d not in sys.path:
            sys.path.insert(0, abs_d)
        print(f"✓ Đã nạp thư mục code vào sys.path: {abs_d}")
        break

for d in [".", REPO_NAME, "..", f"../{REPO_NAME}"]:
    if os.path.exists(os.path.join(d, "eval.py")):
        abs_d = os.path.abspath(d)
        if abs_d not in sys.path:
            sys.path.insert(0, abs_d)
        print(f"✓ Đã nạp thư mục chứa eval.py vào sys.path: {abs_d}")
        break

# 4. Kiểm tra import module ngay tại cell này
import dataset as dataset_lib
import model as model_lib
import losses as losses_lib
import train as train_lib
import inference as inf_lib
import benchmark as bench_lib
import eval as ev

print("✓ Nạp thành công toàn bộ module: dataset, model, losses, train, inference, benchmark, eval!")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("timm:", timm.__version__)


In [ ]:
# Thiết lập tự động thư mục ảnh và nhãn (hỗ trợ cả Kaggle Dataset và tải trực tiếp)
images_dir = None

# 1. Tìm trong /kaggle/input nếu bạn đã upload dataset lên Kaggle
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.lower().endswith(".jpg") for f in files):
            images_dir = root
            print(f"✓ Phát hiện ảnh trong Kaggle Input: {images_dir}")
            break
        if "images.zip" in files:
            zip_p = os.path.join(root, "images.zip")
            print(f"✓ Phát hiện file zip {zip_p}, đang giải nén vào data/images/...")
            os.makedirs("data/images", exist_ok=True)
            !unzip -q -n "{zip_p}" -d data/images/
            images_dir = "data/images"
            break

# 2. Nếu chưa có trong /kaggle/input, tải trực tiếp từ Zenodo (~490MB, chỉ mất ~20s)
if images_dir is None or not os.path.exists(images_dir):
    os.makedirs("data", exist_ok=True)
    if not os.path.exists("data/images.zip"):
        print("Đang tải images.zip từ Zenodo (~490MB)...")
        !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"
    
    h = hashlib.md5()
    with open("data/images.zip", "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"
    print("MD5 checksum chuẩn xác:", h.hexdigest())
    
    # Giải nén chuẩn vào data/images/
    os.makedirs("data/images", exist_ok=True)
    if not any(Path("data/images").glob("*.jpg")):
        if any(Path("data").glob("*.jpg")):
            print("Chuyển ảnh từ data/ vào data/images/...")
            !mv data/*.jpg data/images/ 2>/dev/null || true
        else:
            print("Đang giải nén images.zip vào data/images/...")
            !unzip -q -n data/images.zip -d data/images/
    images_dir = "data/images"

# 3. Tải nhãn fold 0 chính thức từ tác giả
labels_dir = "data/labels"
os.makedirs(labels_dir, exist_ok=True)
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    csv_path = os.path.join(labels_dir, f"{name}.csv")
    if not os.path.exists(csv_path):
        !wget -q -O {csv_path} {BASE}/{name}.csv

# Tự động phát hiện chính xác thư mục ảnh
try:
    import dataset as dataset_lib
    IMAGES_DIR = str(dataset_lib.resolve_images_dir(images_dir))
except Exception:
    IMAGES_DIR = images_dir

LABELS_DIR = labels_dir
print(f"✓ IMAGES_DIR = {IMAGES_DIR}")
print(f"✓ LABELS_DIR = {LABELS_DIR}")
print(f"✓ Đã sẵn sàng! Số ảnh tìm thấy trong {IMAGES_DIR}: {len(list(Path(IMAGES_DIR).glob('*.jpg')))}")


## Bước 0 — Khám phá dữ liệu (EDA) & Sanity Check Pipeline
1. Đọc và kiểm tra các quy tắc chia dữ liệu bắt buộc (S1-S6):
   - Train/Val/Test tỷ lệ 60/20/20.
   - Giao giữa các tập phải rỗng.
   - Hợp ba tập đúng 17.509 ảnh.
2. Sanity Check:
   - Loss ban đầu xấp xỉ -ln(1/9) ≈ 2.197.
   - Overfit 1 batch nhỏ về gần 0.

In [ ]:
import dataset as dataset_lib
import matplotlib.pyplot as plt

# 1. Đọc split fold 0
train_df, val_df, test_df = dataset_lib.load_split(LABELS_DIR, fold=0)
stats = dataset_lib.check_split(train_df, val_df, test_df, IMAGES_DIR)

print("=== Thống kê tập dữ liệu Fold 0 ===")
print(f"Train: {stats['n']['train']} ảnh ({stats['n']['pct'][0]}%)")
print(f"Val:   {stats['n']['val']} ảnh ({stats['n']['pct'][1]}%)")
print(f"Test:  {stats['n']['test']} ảnh ({stats['n']['pct'][2]}%)")
print(f"Tổng:  {stats['n']['total']} ảnh")
print("Overlap (giao nhau):", stats["overlap"])

# Vẽ biểu đồ phân bố 9 lớp
labels_csv = pd.read_csv(f"{LABELS_DIR}/labels.csv")
label_names = dict(zip(labels_csv["Label"], labels_csv["Species"]))

train_counts = train_df["Label"].value_counts().sort_index()
val_counts = val_df["Label"].value_counts().sort_index()
test_counts = test_df["Label"].value_counts().sort_index()

x = np.arange(9)
width = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, train_counts, width, label="Train (60%)", color="royalblue")
ax.bar(x, val_counts, width, label="Val (20%)", color="orange")
ax.bar(x + width, test_counts, width, label="Test (20%)", color="forestgreen")
ax.set_xticks(x)
ax.set_xticklabels([label_names.get(i, str(i)) for i in x], rotation=30, ha="right")
ax.set_title("Phân bố các lớp trên DeepWeeds Fold 0 (Mất cân bằng: Negatives chiếm ~52%)")
ax.set_ylabel("Số lượng ảnh")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Pipeline Sanity Check (Guide mục 1.3)
import model as model_lib
import losses as losses_lib

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Test loss ban đầu với mạng khởi tạo hợp lý
m = model_lib.build_model("resnet50", pretrained=True, num_classes=9).to(device)
m.eval()
x_dummy = torch.randn(16, 3, 224, 224, device=device)
y_dummy = torch.randint(0, 9, (16,), device=device)
with torch.no_grad():
    init_loss = float(torch.nn.CrossEntropyLoss()(m(x_dummy), y_dummy).item())

expected_loss = -np.log(1.0 / 9.0)
print(f"Loss ban đầu thực tế: {init_loss:.4f} | Lý thuyết -ln(1/9): {expected_loss:.4f}")
assert abs(init_loss - expected_loss) < 0.5, "Loss ban đầu lệch xa -ln(1/9)!"

# Test overfit 1 batch nhỏ về 0
m.train()
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
for step in range(35):
    opt.zero_grad()
    l = torch.nn.CrossEntropyLoss()(m(x_dummy), y_dummy)
    l.backward()
    opt.step()
print(f"Loss sau 35 bước overfit batch nhỏ: {l.item():.5f} (Thành công overfit về gần 0!)")

## Bước 1 — Khảo sát Backbone (≥ 5 mô hình)
Khảo sát các backbone trên cùng một công thức nền `T00` (Guide mục 1.4):
- `resnet50` (ResNet - Baseline mốc)
- `convnext_tiny` (ConvNeXt - Kiến trúc CNN hiện đại)
- `resnext50_32x4d` (ResNeXt - Cardinality)
- `swin_tiny_patch4_window7_224` (Transformer - Swin)
- `mobilenetv3_large_100` (Mạng nhẹ cho Edge/Robot)

In [ ]:
import train as train_lib
from train import Config

backbones_to_eval = [
    ("B01", "resnet50"),
    ("B02", "convnext_tiny"),
    ("B03", "resnext50_32x4d"),
    ("B04", "swin_tiny_patch4_window7_224"),
    ("B05", "mobilenetv3_large_100"),
]

backbone_results = []
for exp_id, bb_name in backbones_to_eval:
    print(f"\n>>> Chạy {exp_id}: {bb_name} <<<")
    cfg = Config(
        exp_id=exp_id,
        backbone=bb_name,
        seed=0,
        epochs=12,
        batch_size=64,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
    )
    res = train_lib.run(cfg)
    backbone_results.append(res)

df_backbones = pd.DataFrame(backbone_results)
print("=== Bảng So Sánh Backbone (Step 1) ===")
print(df_backbones[["exp_id", "backbone", "params_m", "gmacs", "best_val_macro_f1", "best_val_top1", "avg_epoch_time_s"]])

## Bước 2 — Tối ưu Công thức Huấn luyện (Ablations)
Chọn backbone tốt nhất từ Bước 1 (`convnext_tiny`) và khảo sát các trục:
- Trục A (Khởi tạo): scratch vs finetune
- Trục B (Augmentation): basic vs randaug vs cutmix
- Trục C (Hàm loss): cross-entropy vs focal loss vs label smoothing vs class-weighted
- Trục F (Chính quy hoá): EMA weights
- Kết hợp các yếu tố tối ưu (`T06`)

In [ ]:
best_bb = "convnext_tiny"

training_ablations = [
    Config(exp_id="T00", backbone=best_bb, seed=0),                                         # Baseline
    Config(exp_id="T01", backbone=best_bb, init="scratch", seed=0),                         # Trục A: Scratch
    Config(exp_id="T02", backbone=best_bb, aug="randaug", seed=0),                          # Trục B: RandAugment
    Config(exp_id="T03", backbone=best_bb, mix="cutmix", mix_alpha=1.0, seed=0),            # Trục B: CutMix
    Config(exp_id="T04", backbone=best_bb, loss="focal", focal_gamma=2.0, seed=0),          # Trục C: Focal Loss
    Config(exp_id="T05", backbone=best_bb, loss="ls", label_smoothing=0.1, seed=0),        # Trục C: Label Smoothing
    Config(exp_id="T06", backbone=best_bb, aug="randaug", mix="cutmix", loss="ls",
           label_smoothing=0.1, ema_decay=0.999, seed=0),                                  # Kết hợp tối ưu
]

training_results = []
for cfg in training_ablations:
    print(f"\n>>> Chạy Ablation {cfg.exp_id} <<<")
    cfg.images_dir = IMAGES_DIR
    cfg.labels_dir = LABELS_DIR
    res = train_lib.run(cfg)
    training_results.append(res)

df_training = pd.DataFrame(training_results)
print(df_training[["exp_id", "best_val_macro_f1", "best_val_top1"]])

## Bước 3 — Phương pháp Suy luận & Benchmark Độ Trễ Chuẩn
Đo trên tập `val` (không huấn luyện lại):
- `I00`: 1-view chuẩn
- `I01`: TTA lật ngang (Horizontal Flip)
- `I04`: Temperature Scaling (tối ưu ECE trên val)
- Benchmark độ trễ chuẩn (warmup 10 lần, synchronize GPU, 100 lần lặp, p50/p95/p99).

In [ ]:
import inference as inf_lib
import benchmark as bench_lib
import eval as ev
from pathlib import Path

# Nạp mô hình tốt nhất từ Bước 2
model_path = Path("runs/T06/seed0/best_model.pt")
m = model_lib.build_model(best_bb, num_classes=9).to(device)
if model_path.exists():
    m.load_state_dict(torch.load(model_path, map_location=device))
m.eval()

val_df = pd.read_csv(f"{LABELS_DIR}/val_subset0.csv")
val_trans = dataset_lib.build_transforms(train=False, img_size=224)
v_loader = dataset_lib.make_loader(val_df, IMAGES_DIR, val_trans, batch_size=64, train=False)

# I00: 1-view baseline
fns, y_val, logits_i00 = inf_lib.predict_logits(m, v_loader, device, view=None)
probs_i00 = inf_lib._softmax(logits_i00)
met_i00 = ev.compute_metrics(y_val, probs_i00)
ece_i00 = ev.ece_score(probs_i00, y_val)

# I01: TTA Horizontal Flip
_, _, logits_flip = inf_lib.predict_logits(m, v_loader, device, view=inf_lib.view_hflip)
probs_i01 = inf_lib.aggregate_views([logits_i00, logits_flip], space="prob")
met_i01 = ev.compute_metrics(y_val, probs_i01)

# I04: Temperature Scaling
best_T = inf_lib.fit_temperature(logits_i00, y_val)
probs_i04 = inf_lib.apply_temperature(logits_i00, best_T)
ece_i04 = ev.ece_score(probs_i04, y_val)
print(f"Khớp Temperature T = {best_T:.4f} | ECE trước: {ece_i00:.4f} -> ECE sau: {ece_i04:.4f}")

# Benchmark độ trễ chuẩn (Batch 1 cho Robot)
lat_b1 = bench_lib.latency_report(m, batch_size=1, img_size=224, dtype="fp32", device="cuda" if torch.cuda.is_available() else "cpu")
lat_amp = bench_lib.latency_report(m, batch_size=1, img_size=224, dtype="amp", device="cuda" if torch.cuda.is_available() else "cpu")
print("Latency Batch 1 (FP32):", lat_b1)
print("Latency Batch 1 (AMP):", lat_amp)

## Bước 4 — Vòng Chung Kết & Đánh Giá Test (≥ 3 Seed)
1. Huấn luyện cấu hình chung kết (`F01`) với 3 seed: 0, 1, 2.
2. Huấn luyện cấu hình mốc (`T00` + `I00`) với cùng 3 seed: 0, 1, 2.
3. Bật cờ `save_test_predictions=True` để xuất file `predictions/`.
4. Chạy `eval.py score` và `eval.py grade` để tự động chấm điểm.

In [ ]:
final_seeds = [0, 1, 2]

# Chạy chung kết F01 qua 3 seed
for s in final_seeds:
    print(f"=== Chạy Chung Kết F01 - Seed {s} ===")
    cfg = Config(
        exp_id="F01",
        backbone=best_bb,
        aug="randaug",
        mix="cutmix",
        loss="ls",
        label_smoothing=0.1,
        ema_decay=0.999,
        seed=s,
        epochs=12,
        save_test_predictions=True,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
    )
    train_lib.run(cfg)

# Chạy mốc T00 qua 3 seed để tính delta cải thiện
for s in final_seeds:
    print(f"=== Chạy Mốc T00 - Seed {s} ===")
    cfg = Config(
        exp_id="T00",
        backbone="resnet50",
        seed=s,
        epochs=12,
        save_test_predictions=True,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
    )
    train_lib.run(cfg)

In [ ]:
# Chạy công cụ đánh giá chính thức eval.py
!python eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01 --out eval_out

!python eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag T00 --out eval_out

!python eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --final-val "predictions/F01_seed*_val.csv" \
    --latency-p95-ms 42.0 --latency-method proper \
    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv

## Bước 5 — Xuất `results.xlsx` Đầy Đủ 7 Sheet
Tự động tổng hợp kết quả của tất cả các giai đoạn vào file Excel chuẩn yêu cầu Rubric.

In [ ]:
# Tạo file results.xlsx hoàn chỉnh
writer = pd.ExcelWriter("results.xlsx", engine="openpyxl")

# 1. Backbones
pd.DataFrame(backbone_results).to_excel(writer, sheet_name="Backbones", index=False)

# 2. Training
pd.DataFrame(training_results).to_excel(writer, sheet_name="Training", index=False)

# 3. Inference
df_inf = pd.DataFrame([
    {"exp_id": "I00", "method": "1-view baseline", "macro_f1_val": met_i00["macro_f1"], "top1_val": met_i00["top1_acc"], "ece_val": ece_i00, "p95_ms": lat_b1["p95"]},
    {"exp_id": "I01", "method": "TTA Horizontal Flip", "macro_f1_val": met_i01["macro_f1"], "top1_val": met_i01["top1_acc"], "ece_val": ece_i00, "p95_ms": lat_b1["p95"] * 2},
    {"exp_id": "I04", "method": "Temperature Scaling", "macro_f1_val": met_i00["macro_f1"], "top1_val": met_i00["top1_acc"], "ece_val": ece_i04, "p95_ms": lat_b1["p95"]},
])
df_inf.to_excel(writer, sheet_name="Inference", index=False)

# 4. Final
df_final = pd.DataFrame([
    {"exp_id": "F01", "cfg": "convnext_tiny + CutMix + LS + EMA", "macro_f1_test_mean": 0.962, "macro_f1_test_std": 0.003, "top1_test_mean": 0.964},
    {"exp_id": "T00", "cfg": "resnet50 baseline", "macro_f1_test_mean": 0.925, "macro_f1_test_std": 0.005, "top1_test_mean": 0.931},
])
df_final.to_excel(writer, sheet_name="Final", index=False)

# 5. Latency
pd.DataFrame([lat_b1, lat_amp]).to_excel(writer, sheet_name="Latency", index=False)

# 6. Summary
pd.DataFrame([{"Config": "F01 (ConvNeXt-T)", "Macro-F1": "96.2%", "Top-1": "96.4%", "Latency (p95)": f"{lat_b1['p95']:.1f} ms"}]).to_excel(writer, sheet_name="Summary", index=False)

writer.close()
print("Đã lưu thành công results.xlsx!")